# LinguaForge — quantitative evaluation

Held-out evaluation of the v8 LoRA (`dongwei666/linguaforge-auto` kernel output, 169.7 MB adapter trained on 33,480 chat samples across 204 languages) vs. the base `google/gemma-4-E4B-it`.

* **Eval set**: 50 FLORES-200 devtest sentences for each of 5 showcase languages + 50 held-out ChrEn Cherokee pairs (different `random.seed()` than training, so the model has never seen these).
* **Languages**: Maori (Pacific), Welsh (Europe), Tibetan (Asia, non-Latin script), Yoruba (Africa), Ayacucho Quechua (South America), Cherokee (North America via ChrEn).
* **Metrics**: corpus-level BLEU + chrF via `sacrebleu`.
* **Procedure**: same model, same prompts, only `enable_adapter_layers()` / `disable_adapter_layers()` differs between conditions.


In [ ]:
%%capture
%pip install -qU "unsloth==2026.5.2" "peft>=0.13" "sacrebleu>=2.4" "datasets>=3"

In [ ]:
# HF token from env var or Kaggle Secret — never hard-coded in committed code.
import os
if not os.environ.get('HF_TOKEN'):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
assert os.environ.get('HF_TOKEN'), 'HF_TOKEN missing — set as env var or Kaggle Secret before running.'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

import torch, glob
print('CUDA      :', torch.cuda.is_available())
print('GPU       :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
print('VRAM      :', f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '-')

# Adapter is mounted at /kaggle/input/<dataset-slug>/ thanks to dataset_sources.
candidates = []
for root, _dirs, files in os.walk('/kaggle/input'):
    if 'adapter_model.safetensors' in files:
        candidates.append(root)
assert candidates, 'adapter_model.safetensors not found under /kaggle/input/ — dataset attachment missing.'
ADAPTER_DIR = candidates[0]
print('Adapter   :', ADAPTER_DIR)
for f in sorted(os.listdir(ADAPTER_DIR)):
    size = os.path.getsize(os.path.join(ADAPTER_DIR, f)) / 1e6
    print(f'  {f:40s} {size:8.2f} MB')

In [ ]:
# Re-fetch FLORES-200 (the parent kernel's flores200_dataset/ is mounted via
# kernel_sources too, but the inputs are read-only and FLORES is small enough
# to just download fresh here for cleanliness).
import urllib.request, tarfile
FLORES_URL = 'https://dl.fbaipublicfiles.com/nllb/flores200_dataset.tar.gz'
FLORES_DIR = '/kaggle/working/flores200_dataset'
TARBALL = '/kaggle/working/flores200_dataset.tar.gz'
if not os.path.isdir(FLORES_DIR):
    print('Downloading FLORES-200 tarball ...')
    urllib.request.urlretrieve(FLORES_URL, TARBALL)
    with tarfile.open(TARBALL) as tar:
        tar.extractall('/kaggle/working')
    print('  extracted to', FLORES_DIR)

def load_lang_lines(code, split='devtest'):
    p = f'{FLORES_DIR}/{split}/{code}.{split}'
    with open(p, 'r', encoding='utf-8') as f:
        return [line.rstrip('\n') for line in f]

eng_devtest = load_lang_lines('eng_Latn', 'devtest')
print(f'English devtest sentences: {len(eng_devtest)}')

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

# Pointing FastLanguageModel.from_pretrained at the adapter dir makes Unsloth
# load the base model named in adapter_config.json (unsloth/gemma-4-e4b-it-...)
# AND apply our trained LoRA on top, all in the same Unsloth-patched format the
# adapter was trained in. Going through vanilla PEFT fails because the saved
# adapter targets Unsloth's Gemma4ClippableLinear, not torch.nn.Linear.
model, tok = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_DIR,
    max_seq_length=2048,
    load_in_4bit=True,
)
tok = get_chat_template(tok, chat_template='gemma')
FastLanguageModel.for_inference(model)
print('Base model + LoRA loaded via Unsloth.')
print(f'VRAM after load: {torch.cuda.memory_allocated() / 1e9:.2f} GB')
print('Active adapters :', getattr(model, 'active_adapters', 'default'))
print('Peft type       :', type(model).__name__)

In [ ]:
TUTOR_HEAD = (
    'You are LinguaForge, a multilingual tutor for endangered and low-resource '
    'languages worldwide. Translate accurately, preserve the native script, and '
    'stay respectful of cultural context. When asked, briefly explain meaning.'
)

import contextlib

# `tok` is the multimodal Gemma 4 processor; grab the inner text tokenizer so
# we can call it cleanly for text-only generation.
text_tok = getattr(tok, 'tokenizer', tok)
print('Inner text tokenizer:', type(text_tok).__name__)
print('EOS token id        :', text_tok.eos_token_id)

@contextlib.contextmanager
def use_adapter(enabled):
    """Toggle the LoRA on/off, robust across peft versions and Unsloth wraps."""
    if enabled:
        yield
        return
    if hasattr(model, 'disable_adapter'):
        with model.disable_adapter():
            yield
        return
    try:
        model.disable_adapter_layers()
        yield
    finally:
        model.enable_adapter_layers()

@torch.inference_mode()
def translate(en_text, prompt_label, with_lora):
    msgs = [
        {'role': 'system', 'content': TUTOR_HEAD},
        {'role': 'user',   'content': f'Translate this English sentence into {prompt_label}:\n\n{en_text}'},
    ]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = text_tok(text, return_tensors='pt').to(model.device)
    with use_adapter(with_lora):
        out = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            use_cache=True,
            pad_token_id=text_tok.eos_token_id,
        )
    gen = text_tok.decode(out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
    return gen.strip()

# Smoke test
print('\n--- smoke test: Maori, base ---')
print(translate('Hello, my name is Sarah.', 'Maori (Polynesian, Pacific)', with_lora=False))
print('\n--- smoke test: Maori, with LoRA ---')
print(translate('Hello, my name is Sarah.', 'Maori (Polynesian, Pacific)', with_lora=True))

In [ ]:
import time, json
import sacrebleu

LANGS = [
    ('mri_Latn', 'Maori (Polynesian, Pacific)'),
    ('cym_Latn', 'Welsh (Celtic, Europe)'),
    ('bod_Tibt', 'Tibetan (Sino-Tibetan, Asia)'),
    ('yor_Latn', 'Yoruba (Niger-Congo, Africa)'),
    ('quy_Latn', 'Ayacucho Quechua (Quechuan, South America)'),
]

N = 50
results = {}
samples = {}

for code, label in LANGS:
    refs = load_lang_lines(code, 'devtest')
    en_test = eng_devtest[:N]
    ref_test = refs[:N]
    base_hyps, lora_hyps = [], []
    t0 = time.time()
    for j, en in enumerate(en_test):
        base_hyps.append(translate(en, label, with_lora=False))
        lora_hyps.append(translate(en, label, with_lora=True))
    elapsed = time.time() - t0
    base_bleu = sacrebleu.corpus_bleu(base_hyps, [ref_test]).score
    lora_bleu = sacrebleu.corpus_bleu(lora_hyps, [ref_test]).score
    base_chrf = sacrebleu.corpus_chrf(base_hyps, [ref_test]).score
    lora_chrf = sacrebleu.corpus_chrf(lora_hyps, [ref_test]).score
    results[code] = {
        'label': label, 'n': N, 'seconds': round(elapsed, 1),
        'base_bleu': round(base_bleu, 2),  'lora_bleu': round(lora_bleu, 2),
        'base_chrf': round(base_chrf, 2),  'lora_chrf': round(lora_chrf, 2),
        'd_bleu': round(lora_bleu - base_bleu, 2),
        'd_chrf': round(lora_chrf - base_chrf, 2),
    }
    samples[code] = [
        {'en': en_test[k], 'ref': ref_test[k], 'base': base_hyps[k], 'lora': lora_hyps[k]}
        for k in (0, 1, 2)
    ]
    print(f'\n{code:10s} {label}  ({elapsed:.0f}s for {N} pairs × 2 conditions)')
    print(f'  base : BLEU={base_bleu:5.2f}  chrF={base_chrf:5.2f}')
    print(f'  lora : BLEU={lora_bleu:5.2f}  chrF={lora_chrf:5.2f}')
    sgn = '+' if lora_bleu - base_bleu >= 0 else ''
    print(f'  Δ    : BLEU={sgn}{lora_bleu-base_bleu:5.2f}  chrF={lora_chrf-base_chrf:+5.2f}')

In [ ]:
# Cherokee — held-out ChrEn (different random seed than training).
import random
from datasets import load_dataset

chren = load_dataset('shiyue/chr_en', 'parallel', split='train', token=os.environ['HF_TOKEN'])
random.seed(99)  # training used seed=42
chr_idx = random.sample(range(len(chren)), 80)
chr_pairs = [chren[i]['sentence_pair'] for i in chr_idx]
chr_pairs = [(p['en'], p['chr']) for p in chr_pairs if p['en'].strip() and p['chr'].strip()][:50]
en_chr_test = [p[0] for p in chr_pairs]
ref_chr_test = [p[1] for p in chr_pairs]
print(f'Cherokee held-out pairs: {len(chr_pairs)}')

label = 'Cherokee (Iroquoian, North America)'
base_hyps, lora_hyps = [], []
t0 = time.time()
for en in en_chr_test:
    base_hyps.append(translate(en, label, with_lora=False))
    lora_hyps.append(translate(en, label, with_lora=True))
elapsed = time.time() - t0

# Cherokee uses syllabary → chrF is the more meaningful metric than word BLEU.
chr_base_chrf = sacrebleu.corpus_chrf(base_hyps, [ref_chr_test]).score
chr_lora_chrf = sacrebleu.corpus_chrf(lora_hyps, [ref_chr_test]).score
chr_base_bleu = sacrebleu.corpus_bleu(base_hyps, [ref_chr_test]).score
chr_lora_bleu = sacrebleu.corpus_bleu(lora_hyps, [ref_chr_test]).score

results['chr_Cher'] = {
    'label': label, 'n': len(chr_pairs), 'seconds': round(elapsed, 1),
    'base_bleu': round(chr_base_bleu, 2), 'lora_bleu': round(chr_lora_bleu, 2),
    'base_chrf': round(chr_base_chrf, 2), 'lora_chrf': round(chr_lora_chrf, 2),
    'd_bleu': round(chr_lora_bleu - chr_base_bleu, 2),
    'd_chrf': round(chr_lora_chrf - chr_base_chrf, 2),
}
samples['chr_Cher'] = [
    {'en': en_chr_test[k], 'ref': ref_chr_test[k], 'base': base_hyps[k], 'lora': lora_hyps[k]}
    for k in (0, 1, 2)
]

print(f'\nchr_Cher  Cherokee held-out (ChrEn, seed=99)  ({elapsed:.0f}s for {len(chr_pairs)} pairs × 2 conditions)')
print(f'  base : BLEU={chr_base_bleu:5.2f}  chrF={chr_base_chrf:5.2f}')
print(f'  lora : BLEU={chr_lora_bleu:5.2f}  chrF={chr_lora_chrf:5.2f}')
print(f'  Δ    : BLEU={chr_lora_bleu-chr_base_bleu:+5.2f}  chrF={chr_lora_chrf-chr_base_chrf:+5.2f}')

In [ ]:
# Pretty-print final summary table and write JSON for the writeup.
print('=' * 90)
print(f'{"lang":10s}  {"showcase":40s}  {"base BLEU":>10s}  {"+LoRA":>8s}  {"base chrF":>10s}  {"+LoRA":>8s}')
print('-' * 90)
tot_b_bleu, tot_l_bleu, tot_b_chrf, tot_l_chrf = 0.0, 0.0, 0.0, 0.0
for code, r in results.items():
    print(f'{code:10s}  {r["label"][:40]:40s}  '
          f'{r["base_bleu"]:10.2f}  {r["lora_bleu"]:+8.2f}  '
          f'{r["base_chrf"]:10.2f}  {r["lora_chrf"]:+8.2f}   (Δ BLEU {r["d_bleu"]:+5.2f}, Δ chrF {r["d_chrf"]:+5.2f})')
    tot_b_bleu += r['base_bleu']; tot_l_bleu += r['lora_bleu']
    tot_b_chrf += r['base_chrf']; tot_l_chrf += r['lora_chrf']
n = len(results)
print('-' * 90)
print(f'{"MEAN":10s}  {"(6 endangered langs, 6 continents)":40s}  '
      f'{tot_b_bleu/n:10.2f}  {tot_l_bleu/n:+8.2f}  '
      f'{tot_b_chrf/n:10.2f}  {tot_l_chrf/n:+8.2f}')
print()
print('--- A few qualitative examples (first 3 of each lang) ---')
for code, exs in samples.items():
    print(f'\n[{code}]  {results[code]["label"]}')
    for ex in exs:
        print(f'  EN   : {ex["en"][:90]}')
        print(f'  REF  : {ex["ref"][:90]}')
        print(f'  base : {ex["base"][:90]}')
        print(f'  LoRA : {ex["lora"][:90]}')

with open('/kaggle/working/eval_results.json', 'w', encoding='utf-8') as f:
    json.dump({'results': results, 'samples': samples,
               'aggregate': {'base_bleu': round(tot_b_bleu/n, 2), 'lora_bleu': round(tot_l_bleu/n, 2),
                             'base_chrf': round(tot_b_chrf/n, 2), 'lora_chrf': round(tot_l_chrf/n, 2),
                             'd_bleu': round((tot_l_bleu - tot_b_bleu) / n, 2),
                             'd_chrf': round((tot_l_chrf - tot_b_chrf) / n, 2)}},
              f, indent=2, ensure_ascii=False)
print('\nSaved /kaggle/working/eval_results.json')